# RoadLens Anyang — RDD2022 도로 파손 탐지 모델 학습 (Google Colab)

이 노트북은 **RDD2022** 데이터로 포트홀·균열 탐지 모델을 학습하고,
로컬 Streamlit 앱에서 쓸 `best.onnx` 를 만든다.

## 실행 전 확인
1. 런타임 유형을 **GPU (T4)** 로 변경한다. `런타임 > 런타임 유형 변경 > 하드웨어 가속기: GPU`
2. 셀을 **위에서 아래 순서대로** 실행한다.
3. Google 계정 인증은 Drive 마운트 셀에서 **브라우저 팝업으로 직접** 수행한다.
   이 노트북은 비밀번호·토큰을 저장하지 않는다.

## 실행 모드
| 모드 | 학습 이미지 | 모델 / epochs | T4 기준 소요 |
|---|---|---|---|
| `smoke_test` | 클래스별 40장 | yolov8n / 3 | 수 분 |
| `prototype_train` | 클래스별 1,500장 | yolov8n / 30 | 30분 ~ 1시간 |
| `accuracy_train` | 상한 없음 (약 12,000장) | yolov8n / 60 | 약 3시간 |
| `full_train` | 상한 없음 | yolov8s / 150 | 여러 세션 |

**기본값은 `accuracy_train`** 이다. 파이프라인을 먼저 확인하고 싶으면
`RUN_MODE = "smoke_test"` 로 바꿔 몇 분 만에 끝까지 돌려 본 뒤 되돌린다.

사용 한도로 런타임이 끊겨도 **11-A 의 체크포인트**로 이어서 학습할 수 있고,
**11-C / 11-D 셀로 다른 계정으로 옮겨** 이어갈 수도 있다.

## 데이터 이용조건 (중요)
RDD2022 의 **미국(United_States) 하위 집합은 Google Street View 이용조건** 문제가 있을 수 있어
`EXCLUDE_COUNTRIES` 기본값으로 제외한다. 각 국가별 자료의 라이선스는 원 배포처에서 직접 확인한다.
- 데이터셋: https://figshare.com/articles/dataset/21431547

## 0. 환경 확인 — GPU 사용 가능 여부

In [ ]:
import subprocess, sys, platform

print("Python :", sys.version.split()[0])
print("Platform:", platform.platform())

GPU_AVAILABLE = False
try:
    out = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=60)
    if out.returncode == 0:
        print(out.stdout)
        GPU_AVAILABLE = True
    else:
        print("nvidia-smi 실행 실패:", out.stderr[:500])
except FileNotFoundError:
    print("nvidia-smi 를 찾을 수 없습니다.")

if not GPU_AVAILABLE:
    print("\n[경고] GPU 를 사용할 수 없습니다.")
    print("      런타임 > 런타임 유형 변경 > 하드웨어 가속기: GPU 로 설정한 뒤 다시 실행하세요.")
    print("      CPU 로도 smoke_test 는 가능하지만 매우 느립니다.")
else:
    print("\nGPU 사용 가능")

## 1. Google Drive 마운트

브라우저 팝업에서 **본인 계정으로 직접 인증**한다. 인증 정보는 저장되지 않는다.

In [ ]:
DRIVE_MOUNTED = False
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_MOUNTED = True
    print("Drive 마운트 완료: /content/drive")
except Exception as e:
    print("Drive 마운트를 건너뜁니다:", e)
    print("Colab 이 아니거나 인증을 취소한 경우입니다. 체크포인트는 /content 에만 저장됩니다.")

## 2. 실행 설정 — 여기만 바꾸면 된다

In [ ]:
# ── 실행 모드 ────────────────────────────────────────────────────────────
# "smoke_test" | "prototype_train" | "accuracy_train" | "full_train"
RUN_MODE = "accuracy_train"

# ── 데이터 확보 방법 ─────────────────────────────────────────────────────
# "drive_zip"        : Drive 에 올려 둔 ZIP 사용 (오프라인/반복 실행에 유리)
# "figshare_partial" : figshare 원본에서 필요한 국가·장수만 부분 내려받기 (권장, smoke_test 용)
# "figshare_full"    : figshare 원본 ZIP 13.26GB 전체 내려받기 (full_train 용)
# "extracted_dir"    : 이미 압축을 푼 폴더 사용
DATA_SOURCE = "figshare_partial"

# (A) Drive 에 올려 둔 RDD2022 ZIP 경로. 예: /content/drive/MyDrive/RDD2022.zip
RDD2022_ZIP_PATHS = [
    "/content/drive/MyDrive/RDD2022.zip",
]

# (B) 이미 압축을 푼 폴더 (DATA_SOURCE="extracted_dir" 일 때)
RDD2022_EXTRACTED_DIR = ""

# (C) figshare 원본
#   https://figshare.com/articles/dataset/21431547  (RDD2022, CC BY 4.0)
#   아래 URL 은 figshare API 로 확인한 실제 파일 주소이다.
FIGSHARE_ZIP_URL = "https://ndownloader.figshare.com/files/38030910"   # 13.26 GB
# 부분 내려받기에 사용할 국가와 국가별 최대 이미지 수
#
# figshare 원본을 직접 조회해 확인한 실제 규모 (train 폴더, 주석이 있는 이미지)
#   국가              ZIP크기   이미지수
#   Japan             1.07GB    10,506
#   Norway           10.61GB     8,161   ← 초고해상도라 용량만 크다
#   India             0.53GB     7,706
#   Czech             0.26GB     2,829
#   China_MotorBike   0.19GB     1,977
#   China_Drone       0.16GB     2,401   ← 드론 촬영(기본 제외)
#   United_States     0.44GB     4,805   ← 라이선스 문제로 기본 제외
#
# 아래 4개국이 용량 대비 이미지 수가 가장 좋다: 2.05GB 로 22,018장.
PARTIAL_COUNTRIES = ["Japan", "India", "Czech", "China_MotorBike"]
# None 이면 해당 국가의 이미지를 전부 사용한다(정확도를 올리려면 None).
PARTIAL_MAX_IMAGES_PER_COUNTRY = None

# 압축 해제 위치 (Colab 로컬 디스크. Drive 에 풀면 매우 느리다)
EXTRACT_ROOT = "/content/rdd2022"

# ── 국가 선택 ────────────────────────────────────────────────────────────
# 제외하는 국가와 그 사유. 화면과 결과 JSON 에 사유가 그대로 남는다.
EXCLUDE_COUNTRIES = {
    "United_States": "Google Street View 이용조건 문제 가능성",
    "China_Drone":   "드론 촬영이라 지상 촬영(휴대폰·차량 카메라) 환경과 다름",
}
INCLUDE_COUNTRIES = []   # 비우면 EXCLUDE 를 뺀 전체 사용

# ── 클래스 ───────────────────────────────────────────────────────────────
# RDD2022 표준 4클래스
CLASS_NAMES = ["D00", "D10", "D20", "D40"]
CLASS_LABELS_KO = {"D00": "종방향 균열", "D10": "횡방향 균열",
                   "D20": "거북등 균열", "D40": "포트홀"}

# ── 학습 하이퍼파라미터 (모드별) ─────────────────────────────────────────
MODE_SETTINGS = {
    # images_per_class: 클래스별로 이 장수를 채우면 더 담지 않는다. None 이면 전부 사용.
    "smoke_test":      {"images_per_class": 40,   "epochs": 3,   "batch": 8,  "imgsz": 320, "model": "yolov8n.pt"},
    "prototype_train": {"images_per_class": 1500, "epochs": 30,  "batch": 16, "imgsz": 640, "model": "yolov8n.pt"},
    # 정확도 개선용. 데이터 상한을 걸지 않아 학습 이미지가 prototype 의 약 4배가 된다.
    # T4 기준 약 3시간. 런타임이 끊겨도 11-A 체크포인트로 이어서 할 수 있다.
    "accuracy_train":  {"images_per_class": None, "epochs": 60,  "batch": 16, "imgsz": 640, "model": "yolov8n.pt"},
    # 더 큰 모델로 최대 성능을 노릴 때. 여러 세션에 걸쳐 돌려야 한다.
    "full_train":      {"images_per_class": None, "epochs": 150, "batch": 16, "imgsz": 640, "model": "yolov8s.pt"},
}
assert RUN_MODE in MODE_SETTINGS, f"RUN_MODE 는 {list(MODE_SETTINGS)} 중 하나여야 합니다."
CFG = dict(MODE_SETTINGS[RUN_MODE])

# 데이터 분할 비율 (촬영 출처/국가 기준으로 분리)
SPLIT_RATIO = {"train": 0.7, "val": 0.15, "test": 0.15}
RANDOM_SEED = 42

# ── 출력 위치 ────────────────────────────────────────────────────────────
WORK_DIR = "/content/roadlens"
YOLO_DATASET_DIR = f"{WORK_DIR}/dataset"
OUTPUT_DIR = f"{WORK_DIR}/outputs"
# 체크포인트는 항상 런타임 안의 같은 경로에 쓴다(경로가 고정되어야 다른 계정에서도
# 그대로 이어서 학습할 수 있다). Drive/로컬로 내보내는 일은 아래 체크포인트 셀이 맡는다.
CHECKPOINT_DIR = f"{WORK_DIR}/runs"
RUN_NAME = f"rdd2022_{RUN_MODE}"

import os, json, random
for d in (WORK_DIR, YOLO_DATASET_DIR, OUTPUT_DIR, CHECKPOINT_DIR, EXTRACT_ROOT):
    os.makedirs(d, exist_ok=True)
random.seed(RANDOM_SEED)

print(json.dumps({"RUN_MODE": RUN_MODE, **CFG,
                  "CHECKPOINT_DIR": CHECKPOINT_DIR,
                  "EXCLUDE_COUNTRIES": EXCLUDE_COUNTRIES,
                  "PARTIAL_COUNTRIES": PARTIAL_COUNTRIES,
                  "PARTIAL_MAX_IMAGES_PER_COUNTRY": PARTIAL_MAX_IMAGES_PER_COUNTRY},
                 indent=2, ensure_ascii=False))

## 3. 패키지 설치

In [ ]:
!pip install -q ultralytics onnx onnxruntime 2>&1 | tail -3

import ultralytics
ultralytics.checks()
print("ultralytics", ultralytics.__version__)

## 4. 데이터 확보

`DATA_SOURCE` 설정에 따라 Drive ZIP / figshare 부분 내려받기 / figshare 전체 내려받기 중 하나를 수행한다.

In [ ]:
import os, zipfile, shutil, time, subprocess
from pathlib import Path

def unzip_with_progress(zip_path, dest, members=None):
    zip_path = Path(zip_path)
    if not zip_path.exists():
        print(f"[건너뜀] 파일 없음: {zip_path}")
        return False
    print(f"압축 해제: {zip_path.name} ({zip_path.stat().st_size/1e9:.2f} GB) -> {dest}")
    t0 = time.time()
    with zipfile.ZipFile(zip_path) as zf:
        names = members if members is not None else zf.namelist()
        for i, m in enumerate(names, 1):
            zf.extract(m, dest)
            if i % 5000 == 0 or i == len(names):
                print(f"  {i:,}/{len(names):,} ({i/len(names)*100:.1f}%)")
    print(f"완료: {time.time()-t0:.0f}초")
    return True


def pick_subset(zip_path, max_images):
    """중첩 ZIP 에서 이미지+주석 쌍을 max_images 개까지 고른다."""
    with zipfile.ZipFile(zip_path) as zf:
        names = zf.namelist()
    imgs = sorted(n for n in names
                  if "/train/images/" in n and n.lower().endswith((".jpg", ".jpeg", ".png")))
    xmls = {Path(n).stem: n for n in names
            if "/train/annotations/" in n and n.lower().endswith(".xml")}
    pairs = [(i, xmls[Path(i).stem]) for i in imgs if Path(i).stem in xmls]
    if max_images:
        pairs = pairs[:max_images]
    return [p for pair in pairs for p in pair], len(imgs)


DATA_ROOT = EXTRACT_ROOT

if DATA_SOURCE == "extracted_dir":
    assert RDD2022_EXTRACTED_DIR, "RDD2022_EXTRACTED_DIR 를 지정하세요."
    DATA_ROOT = RDD2022_EXTRACTED_DIR
    print("이미 압축 해제된 폴더를 사용합니다:", DATA_ROOT)

elif DATA_SOURCE == "drive_zip":
    ok = False
    for zp in RDD2022_ZIP_PATHS:
        ok |= unzip_with_progress(zp, EXTRACT_ROOT)
    if not ok:
        raise FileNotFoundError(
            f"RDD2022 ZIP 을 찾지 못했습니다. 확인한 경로: {RDD2022_ZIP_PATHS}\n"
            "DATA_SOURCE 를 'figshare_partial' 로 바꾸면 Drive 없이 바로 받을 수 있습니다."
        )

elif DATA_SOURCE in ("figshare_partial", "figshare_full"):
    # 바깥 ZIP 은 국가별 ZIP 을 무압축(STORED)으로 담고 있다.
    #   RDD2022/Czech.zip(0.26GB), China_MotorBike(0.19), China_Drone(0.16),
    #   India(0.53), Japan(1.07), United_States(0.44), Norway(10.61)
    # 따라서 필요한 국가 ZIP 만 골라 받는다.
    try:
        from remotezip import RemoteZip
    except ImportError:
        subprocess.run(["pip", "install", "-q", "remotezip"], check=True)
        from remotezip import RemoteZip

    NESTED_DIR = "/content/rdd2022_nested"
    os.makedirs(NESTED_DIR, exist_ok=True)

    print("figshare ZIP 목차를 읽는 중... (전체 13.26GB 를 내려받지 않습니다)")
    with RemoteZip(FIGSHARE_ZIP_URL) as rz:
        nested = {Path(n).stem: n for n in rz.namelist() if n.lower().endswith(".zip")}
        print("포함된 국가 ZIP:")
        for stem, name in nested.items():
            print(f"  {stem:20s} {rz.getinfo(name).file_size/1e9:6.2f} GB")

        if DATA_SOURCE == "figshare_full":
            wanted_countries = [s for s in nested if not any(
                ex.lower().replace("_", "") in s.lower().replace("_", "")
                for ex in EXCLUDE_COUNTRIES)]
        else:
            wanted_countries = [c for c in PARTIAL_COUNTRIES if c in nested]
            missing = [c for c in PARTIAL_COUNTRIES if c not in nested]
            if missing:
                print(f"[주의] ZIP 에 없는 이름은 건너뜁니다: {missing}")
        if not wanted_countries:
            raise RuntimeError(
                f"내려받을 국가가 없습니다. 사용 가능: {sorted(nested)}"
            )

        print(f"\n내려받을 국가: {wanted_countries}")
        for c in wanted_countries:
            dest = Path(NESTED_DIR) / nested[c]
            if dest.exists():
                print(f"  {c}: 이미 있음, 건너뜁니다")
                continue
            t0 = time.time()
            rz.extract(nested[c], NESTED_DIR)
            print(f"  {c}: {dest.stat().st_size/1e9:.2f} GB, {time.time()-t0:.0f}초")

    limit = None if DATA_SOURCE == "figshare_full" else PARTIAL_MAX_IMAGES_PER_COUNTRY
    for c in wanted_countries:
        zpath = Path(NESTED_DIR) / nested[c]
        members, total_imgs = pick_subset(zpath, limit)
        print(f"\n{c}: 전체 train 이미지 {total_imgs:,}장 중 {len(members)//2:,}장 사용")
        unzip_with_progress(zpath, EXTRACT_ROOT, members=members)
        zpath.unlink()  # 디스크 절약

    if DATA_SOURCE == "figshare_partial":
        print("\n[주의] 국가별 앞부분 이미지만 사용하므로 데이터가 편향될 수 있습니다.")
        print("       성능 수치를 제시하려면 figshare_full 또는 drive_zip 으로 다시 학습하세요.")

else:
    raise ValueError(f"알 수 없는 DATA_SOURCE: {DATA_SOURCE}")

print("\nDATA_ROOT =", DATA_ROOT)
for p in sorted(Path(DATA_ROOT).glob("*/*"))[:10]:
    print(" -", p)
du = subprocess.run(["du", "-sh", DATA_ROOT], capture_output=True, text=True)
print("사용 용량:", du.stdout.strip())


## 5. 국가(촬영 출처)별 폴더 탐색 및 미국 데이터 제외

RDD2022 는 `<국가>/train/images`, `<국가>/train/annotations/xmls` 구조를 가진다.
폴더 구조가 달라도 `images` 와 `annotations` 를 재귀 탐색해 찾는다.

In [ ]:
from pathlib import Path
import re

def find_country_dirs(root: str):
    """images 폴더를 가진 상위 디렉터리를 국가(출처) 단위로 수집한다."""
    root = Path(root)
    found = {}
    for img_dir in root.rglob("images"):
        if not img_dir.is_dir():
            continue
        # images 와 같은 레벨의 annotations 를 찾는다
        ann_dir = None
        for cand in (img_dir.parent / "annotations" / "xmls",
                     img_dir.parent / "annotations",
                     img_dir.parent / "xmls"):
            if cand.is_dir():
                ann_dir = cand
                break
        # 국가명 추정: root 바로 아래 첫 번째 경로 조각
        try:
            rel = img_dir.relative_to(root)
            country = rel.parts[0] if len(rel.parts) > 0 else img_dir.parent.name
        except ValueError:
            country = img_dir.parent.name
        found.setdefault(country, []).append((img_dir, ann_dir))
    return found

country_dirs = find_country_dirs(DATA_ROOT)
print("발견된 출처(국가) 폴더:")
for c, pairs in sorted(country_dirs.items()):
    print(f"  {c}: {len(pairs)}개 image 폴더, annotations {sum(1 for _, a in pairs if a)}개")

def excluded_reason(country: str):
    """제외 대상이면 사유를, 아니면 None 을 돌려준다."""
    norm = re.sub(r"[^a-z]", "", country.lower())
    for ex, reason in EXCLUDE_COUNTRIES.items():
        if re.sub(r"[^a-z]", "", ex.lower()) in norm:
            return reason
    return None

selected_countries = []
for c in sorted(country_dirs):
    reason = excluded_reason(c)
    if reason:
        print(f"[제외] {c} — {reason}")
        continue
    if INCLUDE_COUNTRIES and c not in INCLUDE_COUNTRIES:
        print(f"[제외] {c} — INCLUDE_COUNTRIES 에 없음")
        continue
    selected_countries.append(c)

print("\n학습에 사용할 출처:", selected_countries)
if not selected_countries:
    raise RuntimeError("사용할 국가 폴더가 없습니다. EXCLUDE_COUNTRIES / INCLUDE_COUNTRIES 를 확인하세요.")

## 6. 주석 파일 수집 · 손상 파일 검사

In [ ]:
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from PIL import Image

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}

records = []          # (country, image_path, xml_path)
missing_ann = []      # 주석 없는 이미지
orphan_ann = []       # 이미지 없는 주석
corrupt_images = []   # 열리지 않는 이미지
corrupt_xmls = []     # 파싱 실패 XML

for country in selected_countries:
    for img_dir, ann_dir in country_dirs[country]:
        images = {p.stem: p for p in img_dir.iterdir() if p.suffix in IMAGE_EXTS}
        anns = {}
        if ann_dir is not None:
            anns = {p.stem: p for p in ann_dir.rglob("*.xml")}
        for stem, img_path in images.items():
            xml_path = anns.get(stem)
            if xml_path is None:
                missing_ann.append(str(img_path))
                continue
            records.append((country, img_path, xml_path))
        for stem, xml_path in anns.items():
            if stem not in images:
                orphan_ann.append(str(xml_path))

print(f"이미지-주석 쌍: {len(records):,}")
print(f"주석 없는 이미지: {len(missing_ann):,}")
print(f"이미지 없는 주석: {len(orphan_ann):,}")

# 손상 검사 (smoke_test 에서는 표본만 검사해 시간을 아낀다)
check_list = records if RUN_MODE in ("accuracy_train", "full_train") else records[:2000]
print(f"\n손상 검사 대상: {len(check_list):,}건")
for i, (country, img_path, xml_path) in enumerate(check_list, 1):
    try:
        with Image.open(img_path) as im:
            im.verify()
    except Exception as e:
        corrupt_images.append((str(img_path), str(e)))
    try:
        ET.parse(xml_path)
    except Exception as e:
        corrupt_xmls.append((str(xml_path), str(e)))
    if i % 1000 == 0:
        print(f"  {i:,}/{len(check_list):,}")

print(f"\n손상 이미지: {len(corrupt_images)}건")
for p, e in corrupt_images[:5]:
    print("  -", p, e[:80])
print(f"파싱 실패 XML: {len(corrupt_xmls)}건")
for p, e in corrupt_xmls[:5]:
    print("  -", p, e[:80])

bad = {p for p, _ in corrupt_images} | {p for p, _ in corrupt_xmls}
records = [r for r in records if str(r[1]) not in bad and str(r[2]) not in bad]
print(f"\n검사 후 사용 가능한 쌍: {len(records):,}")

## 7. 클래스 분포 확인 (Pascal VOC XML 파싱)

In [ ]:
def parse_voc(xml_path):
    """VOC XML -> (width, height, [(class_name, xmin, ymin, xmax, ymax), ...])"""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    size = root.find("size")
    if size is None:
        return None, None, []
    w = int(float(size.findtext("width", "0")))
    h = int(float(size.findtext("height", "0")))
    boxes = []
    for obj in root.findall("object"):
        name = (obj.findtext("name") or "").strip()
        bnd = obj.find("bndbox")
        if bnd is None:
            continue
        try:
            xmin = float(bnd.findtext("xmin", "0")); ymin = float(bnd.findtext("ymin", "0"))
            xmax = float(bnd.findtext("xmax", "0")); ymax = float(bnd.findtext("ymax", "0"))
        except ValueError:
            continue
        boxes.append((name, xmin, ymin, xmax, ymax))
    return w, h, boxes

class_counter = Counter()
country_counter = Counter()
image_classes = {}          # image_path -> set(classes)
records_with_boxes = []

for country, img_path, xml_path in records:
    w, h, boxes = parse_voc(xml_path)
    kept = [b for b in boxes if b[0] in CLASS_NAMES]
    if w and h and kept:
        records_with_boxes.append((country, img_path, xml_path, w, h, kept))
        for name, *_ in kept:
            class_counter[name] += 1
        country_counter[country] += 1
        image_classes[str(img_path)] = {b[0] for b in kept}

print(f"객체가 있는 이미지: {len(records_with_boxes):,}")
print("\n클래스별 객체 수:")
for c in CLASS_NAMES:
    print(f"  {c} ({CLASS_LABELS_KO[c]}): {class_counter.get(c, 0):,}")
print("\n출처(국가)별 이미지 수:")
for c, n in country_counter.most_common():
    print(f"  {c}: {n:,}")

other = Counter()
for country, img_path, xml_path in records[:5000]:
    _, _, boxes = parse_voc(xml_path)
    for name, *_ in boxes:
        if name not in CLASS_NAMES:
            other[name] += 1
if other:
    print("\n[참고] CLASS_NAMES 밖의 라벨(학습에서 제외됨):", dict(other.most_common(10)))

## 8. 학습/검증/시험 분할

**촬영 출처(국가) 기준으로 분리**한다. 같은 출처의 이미지가 train/val/test 에 섞이면
성능이 실제보다 높게 나올 수 있다. 출처가 1개뿐이면 이미지 단위 무작위 분할로 내려가고
그 사실을 출력한다.

In [ ]:
import random
random.seed(RANDOM_SEED)

# 모드별 표본 축소 (클래스별 상한)
def subsample(recs, per_class):
    if per_class is None:
        return recs
    picked, counts = [], Counter()
    shuffled = recs[:]
    random.shuffle(shuffled)
    for r in shuffled:
        classes = {b[0] for b in r[5]}
        if any(counts[c] < per_class for c in classes):
            picked.append(r)
            for c in classes:
                counts[c] += 1
    return picked

sampled = subsample(records_with_boxes, CFG["images_per_class"])
print(f"표본 축소: {len(records_with_boxes):,} -> {len(sampled):,} (클래스별 상한 {CFG['images_per_class']})")

by_country = defaultdict(list)
for r in sampled:
    by_country[r[0]].append(r)

splits = {"train": [], "val": [], "test": []}
if len(by_country) >= 3:
    countries = sorted(by_country, key=lambda c: -len(by_country[c]))
    n_val = max(1, round(len(countries) * SPLIT_RATIO["val"]))
    n_test = max(1, round(len(countries) * SPLIT_RATIO["test"]))
    test_c = countries[-n_test:]
    val_c = countries[-(n_test + n_val):-n_test]
    train_c = countries[:-(n_test + n_val)]
    for c in train_c: splits["train"] += by_country[c]
    for c in val_c:   splits["val"]   += by_country[c]
    for c in test_c:  splits["test"]  += by_country[c]
    print(f"출처 기준 분할 — train:{train_c} val:{val_c} test:{test_c}")
    SPLIT_STRATEGY = "country_holdout"
else:
    print(f"[주의] 출처가 {len(by_country)}개뿐이라 출처 기준 분할이 불가능합니다.")
    print("       이미지 단위 무작위 분할로 진행합니다. 같은 도로/구간이 섞일 수 있어")
    print("       실제 일반화 성능보다 높게 측정될 수 있습니다.")
    pool = sampled[:]
    random.shuffle(pool)
    n = len(pool)
    n_tr = int(n * SPLIT_RATIO["train"]); n_va = int(n * SPLIT_RATIO["val"])
    splits["train"] = pool[:n_tr]
    splits["val"] = pool[n_tr:n_tr + n_va]
    splits["test"] = pool[n_tr + n_va:]
    SPLIT_STRATEGY = "random_image"

for k, v in splits.items():
    print(f"  {k}: {len(v):,}장")
assert len(splits["train"]) > 0 and len(splits["val"]) > 0, "train/val 이 비어 있습니다. 표본 수를 늘리세요."

## 9. Pascal VOC XML → YOLO 형식 변환

In [ ]:
import shutil
from pathlib import Path

CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}

def voc_box_to_yolo(box, w, h):
    name, xmin, ymin, xmax, ymax = box
    xmin = max(0.0, min(xmin, w)); xmax = max(0.0, min(xmax, w))
    ymin = max(0.0, min(ymin, h)); ymax = max(0.0, min(ymax, h))
    bw, bh = xmax - xmin, ymax - ymin
    if bw <= 1 or bh <= 1:
        return None
    return (CLASS_TO_ID[name], ((xmin + xmax) / 2) / w, ((ymin + ymax) / 2) / h, bw / w, bh / h)

# 기존 데이터셋 폴더 초기화
if Path(YOLO_DATASET_DIR).exists():
    shutil.rmtree(YOLO_DATASET_DIR)
for split in splits:
    (Path(YOLO_DATASET_DIR) / split / "images").mkdir(parents=True, exist_ok=True)
    (Path(YOLO_DATASET_DIR) / split / "labels").mkdir(parents=True, exist_ok=True)

converted = Counter()
skipped_boxes = 0
for split, recs in splits.items():
    for country, img_path, xml_path, w, h, boxes in recs:
        stem = f"{country}_{img_path.stem}"
        dst_img = Path(YOLO_DATASET_DIR) / split / "images" / f"{stem}{img_path.suffix}"
        dst_lbl = Path(YOLO_DATASET_DIR) / split / "labels" / f"{stem}.txt"
        lines = []
        for b in boxes:
            y = voc_box_to_yolo(b, w, h)
            if y is None:
                skipped_boxes += 1
                continue
            cid, cx, cy, bw, bh = y
            lines.append(f"{cid} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
        if not lines:
            continue
        try:
            os.symlink(img_path, dst_img)
        except (OSError, NotImplementedError, FileExistsError):
            shutil.copy2(img_path, dst_img)
        dst_lbl.write_text("\n".join(lines))
        converted[split] += 1

print("YOLO 형식 변환 결과:")
for split in ("train", "val", "test"):
    print(f"  {split}: {converted[split]:,}장")
print(f"크기가 너무 작아 제외한 박스: {skipped_boxes:,}개")

data_yaml = Path(YOLO_DATASET_DIR) / "data.yaml"
data_yaml.write_text(
    f"path: {YOLO_DATASET_DIR}\n"
    f"train: train/images\n"
    f"val: val/images\n"
    f"test: test/images\n"
    f"nc: {len(CLASS_NAMES)}\n"
    f"names: {CLASS_NAMES}\n"
)
print("\ndata.yaml\n" + data_yaml.read_text())

## 10. 변환 결과 검증 (분할별 클래스 분포)

In [ ]:
split_class_counts = {}
for split in ("train", "val", "test"):
    counter = Counter()
    lbl_dir = Path(YOLO_DATASET_DIR) / split / "labels"
    for lbl in lbl_dir.glob("*.txt"):
        for line in lbl.read_text().splitlines():
            if line.strip():
                counter[CLASS_NAMES[int(line.split()[0])]] += 1
    split_class_counts[split] = dict(counter)

import pandas as pd
dist = pd.DataFrame(split_class_counts).fillna(0).astype(int)
dist.index = [f"{c} ({CLASS_LABELS_KO[c]})" for c in dist.index]
print("분할별 클래스(객체) 분포:")
print(dist.to_string())

if (dist["val"] == 0).any():
    print("\n[주의] val 에 객체가 0개인 클래스가 있습니다. 해당 클래스의 지표는 신뢰할 수 없습니다.")

## 11-A. 체크포인트 지속성 — 사용 한도·계정 이동 대비

Colab 런타임이 종료되면 `/content` 안의 파일은 **전부 사라진다.** 그래서 학습 중간에
`last.pt` 를 zip 번들로 묶어 런타임 밖으로 복제해 둔다.

| 미러링 방식 | 조건 | 안전도 |
|---|---|---|
| `drive` | 1번 셀에서 Drive 마운트 | 가장 안전. 런타임이 죽어도 남는다 |
| `download` | Chrome 의 다중 다운로드 허용 | 로컬 PC 에 남는다. 계정과 무관 |
| `none` | — | 런타임 종료 시 전부 소실 |

**다른 계정으로 옮기는 순서**
1. (현재 계정) 11-C 셀로 번들 zip 을 내려받는다.
2. (새 계정) 이 노트북을 업로드하고 **0~10번 셀을 그대로 실행**한다.
   `RANDOM_SEED`·`PARTIAL_COUNTRIES`·`PARTIAL_MAX_IMAGES_PER_COUNTRY` 를 바꾸지 않으면
   같은 데이터셋이 다시 만들어진다.
3. (새 계정) 11-D 셀에서 번들 zip 을 올린다. 설정이 다르면 경고가 뜬다.
4. (새 계정) 학습 셀에서 `RESUME = True` 로 바꿔 실행한다.

In [ ]:
import json, shutil, time, zipfile
from pathlib import Path

RUN_DIR_EXPECTED = Path(CHECKPOINT_DIR) / RUN_NAME
BUNDLE_NAME = f"{RUN_NAME}_resume_bundle.zip"
BUNDLE_PATH = Path(WORK_DIR) / BUNDLE_NAME

# 미러 위치: Drive 를 마운트했으면 Drive, 아니면 없음
MIRROR_DIR = "/content/drive/MyDrive/roadlens_anyang/checkpoints" if DRIVE_MOUNTED else None

# "drive"    : 매 N 에포크마다 Drive 로 복사 (Drive 마운트 필요, 가장 안전)
# "download" : 매 N 에포크마다 브라우저로 내려받기 (Chrome 의 다중 다운로드 허용 필요)
# "none"     : 미러링하지 않음 (런타임이 죽으면 전부 소실)
MIRROR_MODE = "drive" if DRIVE_MOUNTED else "download"
MIRROR_EVERY_N_EPOCHS = 10


def make_resume_bundle(dest=None, quiet=False):
    """last.pt 와 학습 상태를 하나의 zip 으로 묶는다.

    이 zip 만 있으면 다른 계정·다른 런타임에서 같은 지점부터 이어서 학습할 수 있다.
    """
    dest = Path(dest) if dest else BUNDLE_PATH
    run_dir = RUN_DIR_EXPECTED
    last = run_dir / "weights" / "last.pt"
    if not last.exists():
        if not quiet:
            print(f"[건너뜀] 아직 체크포인트가 없습니다: {last}")
        return None

    meta = {
        "saved_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "run_name": RUN_NAME,
        "run_mode": RUN_MODE,
        "cfg": CFG,
        "random_seed": RANDOM_SEED,
        "split_ratio": SPLIT_RATIO,
        # 아래 값이 달라지면 데이터셋이 달라져 이어서 학습한 결과가 무의미해진다.
        "data_source": DATA_SOURCE,
        "partial_countries": PARTIAL_COUNTRIES,
        "partial_max_images_per_country": PARTIAL_MAX_IMAGES_PER_COUNTRY,
        "exclude_countries": EXCLUDE_COUNTRIES,
        "checkpoint_dir": str(CHECKPOINT_DIR),
        "yolo_dataset_dir": str(YOLO_DATASET_DIR),
    }
    (run_dir / "roadlens_resume_meta.json").write_text(
        json.dumps(meta, indent=2, ensure_ascii=False)
    )

    tmp = dest.with_suffix(".tmp.zip")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as zf:
        for rel in ("weights/last.pt", "weights/best.pt", "args.yaml",
                    "results.csv", "roadlens_resume_meta.json"):
            f = run_dir / rel
            if f.exists():
                zf.write(f, f"{RUN_NAME}/{rel}")
    tmp.replace(dest)  # 쓰다 만 파일이 남지 않도록 원자적으로 교체
    if not quiet:
        print(f"번들 생성: {dest} ({dest.stat().st_size/1e6:.1f} MB)")
    return dest


def restore_resume_bundle(bundle_path):
    """다른 계정/런타임에서 받은 번들을 복원한다. 설정이 다르면 경고한다."""
    bundle_path = Path(bundle_path)
    if not bundle_path.exists():
        print(f"[실패] 번들 파일이 없습니다: {bundle_path}")
        return False

    Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundle_path) as zf:
        zf.extractall(CHECKPOINT_DIR)

    meta_file = RUN_DIR_EXPECTED / "roadlens_resume_meta.json"
    if meta_file.exists():
        meta = json.loads(meta_file.read_text())
        print("복원한 학습 정보:")
        print(json.dumps(meta, indent=2, ensure_ascii=False))
        mismatches = []
        if meta.get("cfg", {}).get("epochs") != CFG["epochs"]:
            mismatches.append("epochs")
        if meta.get("cfg", {}).get("imgsz") != CFG["imgsz"]:
            mismatches.append("imgsz")
        if meta.get("partial_countries") != PARTIAL_COUNTRIES:
            mismatches.append("PARTIAL_COUNTRIES")
        if meta.get("partial_max_images_per_country") != PARTIAL_MAX_IMAGES_PER_COUNTRY:
            mismatches.append("PARTIAL_MAX_IMAGES_PER_COUNTRY")
        if meta.get("random_seed") != RANDOM_SEED:
            mismatches.append("RANDOM_SEED")
        if mismatches:
            print("\n[경고] 저장 당시와 다른 설정: " + ", ".join(mismatches))
            print("       같은 값으로 맞추지 않으면 데이터셋이 달라져 이어서 학습한 결과가 무의미합니다.")
        else:
            print("\n설정이 저장 당시와 일치합니다. 이어서 학습할 수 있습니다.")
    else:
        print("[주의] 메타데이터가 없는 번들입니다. 설정이 같은지 직접 확인하세요.")

    last = RUN_DIR_EXPECTED / "weights" / "last.pt"
    print(f"\n복원 위치: {RUN_DIR_EXPECTED}")
    print(f"last.pt 존재: {last.exists()}")
    print("다음 단계: 학습 셀에서 RESUME = True 로 바꾸고 실행하세요.")
    return last.exists()


def _download_bundle(path):
    try:
        from google.colab import files
        files.download(str(path))
        return True
    except Exception as exc:
        print("  자동 다운로드 실패:", exc)
        return False


def attach_checkpoint_mirror(model, every_n_epochs=None, mode=None):
    """에포크마다 체크포인트를 런타임 밖으로 내보내는 콜백을 건다.

    Colab 사용 한도로 런타임이 갑자기 종료되어도 마지막 미러 시점부터 재개할 수 있다.
    """
    every_n_epochs = every_n_epochs or MIRROR_EVERY_N_EPOCHS
    mode = mode or MIRROR_MODE
    state = {"epoch": 0}

    def _mirror(trainer):
        state["epoch"] += 1
        if state["epoch"] % every_n_epochs != 0:
            return
        try:
            bundle = make_resume_bundle(quiet=True)
            if bundle is None:
                return
            size = bundle.stat().st_size / 1e6
            if mode == "drive" and MIRROR_DIR:
                Path(MIRROR_DIR).mkdir(parents=True, exist_ok=True)
                shutil.copy2(bundle, Path(MIRROR_DIR) / bundle.name)
                print(f"[체크포인트] {state['epoch']} 에포크 → Drive 저장 ({size:.1f} MB)")
            elif mode == "download":
                print(f"[체크포인트] {state['epoch']} 에포크 → 브라우저 다운로드 시도 ({size:.1f} MB)")
                _download_bundle(bundle)
            else:
                print(f"[체크포인트] {state['epoch']} 에포크 → {bundle} (런타임 내부에만 있음)")
        except Exception as exc:
            print("[체크포인트] 미러링 실패:", exc)

    model.add_callback("on_fit_epoch_end", _mirror)
    return model


print(f"체크포인트 저장 위치 : {RUN_DIR_EXPECTED}")
print(f"번들 파일           : {BUNDLE_PATH}")
print(f"미러링 방식         : {MIRROR_MODE} (매 {MIRROR_EVERY_N_EPOCHS} 에포크)")
if MIRROR_MODE == "drive":
    print(f"미러 위치           : {MIRROR_DIR}")
elif MIRROR_MODE == "download":
    print("미러 위치           : 브라우저 다운로드 폴더")
    print("  ※ Chrome 이 '여러 파일 다운로드'를 차단하면 주소창 아이콘에서 허용하세요.")
    print("  ※ Drive 를 마운트하면(1번 셀) 더 안전하게 Drive 에 저장됩니다.")
else:
    print("  ※ 미러링을 끄면 런타임 종료 시 학습 진행분이 모두 사라집니다.")

## 11-B. 학습

체크포인트는 매 epoch 저장되고, **기본 10 epoch 마다 런타임 밖(Drive 또는 다운로드)으로 복제**된다.
Colab 사용 한도로 런타임이 갑자기 끊겨도 마지막 복제 시점부터 다시 시작할 수 있다.

- 같은 런타임에서 중단됐다면: 아래 `RESUME = True` 로 바꿔 이 셀만 다시 실행
- 런타임이 죽었거나 **다른 계정으로 옮겼다면**: 11-D 셀로 번들을 복원한 뒤 `RESUME = True`

In [ ]:
import torch
from ultralytics import YOLO

# 학습이 중단된 뒤 이어서 하려면 True 로 바꾼다.
#  - 같은 런타임에서 중단됐다면 그대로 True 로 두고 이 셀만 다시 실행
#  - 런타임이 죽었거나 다른 계정으로 옮겼다면, 먼저 아래 "체크포인트 가져오기" 셀로
#    번들을 복원한 뒤 True 로 바꾼다.
RESUME = False

device = 0 if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu":
    print("[경고] CPU 학습은 매우 느립니다. smoke_test 외에는 권장하지 않습니다.")

last_ckpt = RUN_DIR_EXPECTED / "weights" / "last.pt"

if RESUME and last_ckpt.exists():
    print("체크포인트에서 재개:", last_ckpt)
    model = YOLO(str(last_ckpt))
    train_kwargs = {"resume": True}
else:
    if RESUME:
        print(f"[주의] 재개할 체크포인트가 없어 처음부터 학습합니다: {last_ckpt}")
        print("       다른 계정에서 옮겨왔다면 '체크포인트 가져오기' 셀을 먼저 실행하세요.")
    model = YOLO(CFG["model"])
    train_kwargs = {
        "data": str(data_yaml),
        "epochs": CFG["epochs"],
        "batch": CFG["batch"],
        "imgsz": CFG["imgsz"],
        "project": CHECKPOINT_DIR,
        "name": RUN_NAME,
        "exist_ok": True,
        "save_period": 1,       # 매 epoch 체크포인트 저장
        "patience": 20,
        "seed": RANDOM_SEED,
        "device": device,
        "workers": 2,
        "verbose": True,
    }

# 사용 한도로 런타임이 끊겨도 되살릴 수 있도록 주기적으로 체크포인트를 밖으로 내보낸다.
attach_checkpoint_mirror(model)

results = model.train(**train_kwargs)
RUN_DIR = Path(results.save_dir) if hasattr(results, "save_dir") else RUN_DIR_EXPECTED
print("\n학습 산출물 경로:", RUN_DIR)

# 학습이 끝나면 마지막 상태를 한 번 더 저장해 둔다.
make_resume_bundle()

## 11-C. 체크포인트 번들 내려받기 (수동)

지금까지의 학습 상태를 zip 하나로 묶어 내려받는다. 사용 한도가 얼마 남지 않았을 때 실행한다.

In [ ]:
# 지금 시점의 체크포인트를 번들로 만들어 내려받는다.
# 이 zip 하나만 있으면 다른 계정에서 이어서 학습할 수 있다.
bundle = make_resume_bundle()

if bundle is not None:
    if MIRROR_DIR:
        Path(MIRROR_DIR).mkdir(parents=True, exist_ok=True)
        shutil.copy2(bundle, Path(MIRROR_DIR) / bundle.name)
        print("Drive 에도 저장:", Path(MIRROR_DIR) / bundle.name)
    _download_bundle(bundle)
    print("\n번들에 들어 있는 것:")
    with zipfile.ZipFile(bundle) as zf:
        for info in zf.infolist():
            print(f"  {info.filename}  ({info.file_size/1e6:.2f} MB)")

## 11-D. 다른 계정에서 이어서 학습하기

새 계정의 런타임에서 **0~10번 셀을 먼저 실행해 데이터셋을 같게 만든 뒤** 이 셀을 실행한다.
그다음 학습 셀에서 `RESUME = True` 로 바꾸면 중단 지점부터 이어진다.

In [ ]:
# 다른 계정/런타임에서 만든 번들을 이 런타임으로 가져온다.
# 아래 둘 중 하나를 쓴다.
#   (A) 브라우저에서 zip 파일을 직접 올린다  -> UPLOAD_FROM_BROWSER = True
#   (B) Drive 나 런타임 안의 경로를 지정한다 -> UPLOAD_FROM_BROWSER = False + BUNDLE_SOURCE
UPLOAD_FROM_BROWSER = True
BUNDLE_SOURCE = f"/content/drive/MyDrive/roadlens_anyang/checkpoints/{BUNDLE_NAME}"

source = None
if UPLOAD_FROM_BROWSER:
    try:
        from google.colab import files
        uploaded = files.upload()          # 파일 선택 창이 뜬다
        if uploaded:
            name = next(iter(uploaded))
            source = Path("/content") / name
            Path(source).write_bytes(uploaded[name])
            print("업로드됨:", source)
    except Exception as exc:
        print("업로드를 사용할 수 없습니다:", exc)
else:
    source = BUNDLE_SOURCE

if source:
    restore_resume_bundle(source)
else:
    print("가져올 번들이 없습니다.")

## 12. 검증 — Precision / Recall / mAP@0.5 / mAP@0.5:0.95

In [ ]:
best_pt = Path(RUN_DIR) / "weights" / "best.pt"
assert best_pt.exists(), f"best.pt 가 없습니다: {best_pt}"
print("가중치:", best_pt)

best_model = YOLO(str(best_pt))
val_metrics = best_model.val(data=str(data_yaml), split="val", imgsz=CFG["imgsz"],
                             project=OUTPUT_DIR, name=f"{RUN_NAME}_val", exist_ok=True)

box = val_metrics.box
overall = {
    "precision": float(box.mp),
    "recall": float(box.mr),
    "mAP50": float(box.map50),
    "mAP50_95": float(box.map),
}
print("\n=== 검증(val) 실측값 ===")
for k, v in overall.items():
    print(f"  {k}: {v:.4f}")

# 클래스별 결과
per_class = []
names = best_model.names
for i, cidx in enumerate(box.ap_class_index):
    cname = names[int(cidx)]
    p, r, ap50, ap = box.class_result(i)
    per_class.append({
        "class": cname,
        "class_ko": CLASS_LABELS_KO.get(cname, cname),
        "precision": float(p), "recall": float(r),
        "mAP50": float(ap50), "mAP50_95": float(ap),
    })

per_class_df = pd.DataFrame(per_class)
print("\n=== 클래스별 결과 ===")
print(per_class_df.to_string(index=False) if len(per_class_df) else "(없음)")
print("\n※ 위 숫자는 이번 실행에서 실제로 측정된 값이다. 임의로 수정하지 말 것.")

## 13. 시험(test) 세트 평가 · confusion matrix 저장

In [ ]:
test_images = list((Path(YOLO_DATASET_DIR) / "test" / "images").glob("*"))
test_overall, test_per_class = None, []

if len(test_images) == 0:
    print("[건너뜀] test 세트가 비어 있습니다.")
else:
    test_metrics = best_model.val(data=str(data_yaml), split="test", imgsz=CFG["imgsz"],
                                  project=OUTPUT_DIR, name=f"{RUN_NAME}_test", exist_ok=True)
    tbox = test_metrics.box
    test_overall = {
        "precision": float(tbox.mp), "recall": float(tbox.mr),
        "mAP50": float(tbox.map50), "mAP50_95": float(tbox.map),
    }
    print("=== 시험(test) 실측값 ===")
    for k, v in test_overall.items():
        print(f"  {k}: {v:.4f}")
    for i, cidx in enumerate(tbox.ap_class_index):
        cname = best_model.names[int(cidx)]
        p, r, ap50, ap = tbox.class_result(i)
        test_per_class.append({"class": cname, "class_ko": CLASS_LABELS_KO.get(cname, cname),
                               "precision": float(p), "recall": float(r),
                               "mAP50": float(ap50), "mAP50_95": float(ap)})
    print()
    print(pd.DataFrame(test_per_class).to_string(index=False) if test_per_class else "(클래스별 결과 없음)")

# confusion matrix 이미지 수집 (ultralytics 가 val 디렉터리에 자동 저장)
import glob, shutil
for src in glob.glob(f"{OUTPUT_DIR}/**/confusion_matrix*.png", recursive=True):
    dst = Path(OUTPUT_DIR) / f"{RUN_NAME}_{Path(src).parent.name}_{Path(src).name}"
    shutil.copy2(src, dst)
    print("confusion matrix 저장:", dst)

# confusion matrix 원자료도 CSV 로 남긴다
try:
    cm = val_metrics.confusion_matrix.matrix
    labels = list(CLASS_NAMES) + ["background"]
    cm_df = pd.DataFrame(cm, index=labels[:cm.shape[0]], columns=labels[:cm.shape[1]])
    cm_path = Path(OUTPUT_DIR) / f"{RUN_NAME}_confusion_matrix_val.csv"
    cm_df.to_csv(cm_path, encoding="utf-8-sig")
    print("confusion matrix CSV:", cm_path)
    print(cm_df.to_string())
except Exception as e:
    print("confusion matrix CSV 저장 실패:", e)

## 14. 시험 이미지 추론 결과 저장

In [ ]:
import random
pred_dir = Path(OUTPUT_DIR) / f"{RUN_NAME}_test_predictions"
pred_dir.mkdir(parents=True, exist_ok=True)

sample_imgs = test_images if len(test_images) <= 20 else random.sample(test_images, 20)
if not sample_imgs:
    print("[건너뜀] 추론할 시험 이미지가 없습니다.")
else:
    preds = best_model.predict([str(p) for p in sample_imgs], imgsz=CFG["imgsz"],
                               conf=0.25, save=False, verbose=False)
    rows = []
    import cv2
    for img_path, res in zip(sample_imgs, preds):
        annotated = res.plot()
        out_path = pred_dir / f"pred_{img_path.name}"
        cv2.imwrite(str(out_path), annotated)
        for b in res.boxes:
            rows.append({
                "image": img_path.name,
                "class": best_model.names[int(b.cls.item())],
                "confidence": float(b.conf.item()),
                "x1": float(b.xyxy[0][0]), "y1": float(b.xyxy[0][1]),
                "x2": float(b.xyxy[0][2]), "y2": float(b.xyxy[0][3]),
            })
    pd.DataFrame(rows).to_csv(pred_dir / "predictions.csv", index=False, encoding="utf-8-sig")
    print(f"추론 이미지 {len(sample_imgs)}장, 탐지 {len(rows)}건 저장 -> {pred_dir}")

## 15. 실측 결과 JSON / CSV 내보내기

**이 셀은 측정된 값만 기록한다. 값을 손으로 고치지 말 것.**

In [ ]:
import json, platform
from datetime import datetime, timezone

metrics_payload = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "run_mode": RUN_MODE,
    "run_name": RUN_NAME,
    "split_strategy": SPLIT_STRATEGY,
    "excluded_countries": EXCLUDE_COUNTRIES,   # {국가: 제외 사유}
    "partial_countries": PARTIAL_COUNTRIES,
    "partial_max_images_per_country": PARTIAL_MAX_IMAGES_PER_COUNTRY,
    "countries_used": selected_countries,
    "model_arch": CFG["model"],
    "epochs": CFG["epochs"],
    "batch": CFG["batch"],
    "imgsz": CFG["imgsz"],
    "images_per_class_cap": CFG["images_per_class"],
    "dataset_counts": {k: int(v) for k, v in converted.items()},
    "class_object_counts_by_split": split_class_counts,
    "val_overall": overall,
    "val_per_class": per_class,
    "test_overall": test_overall,
    "test_per_class": test_per_class,
    "environment": {"python": platform.python_version(),
                    "ultralytics": ultralytics.__version__,
                    "torch": torch.__version__,
                    "device": str(device)},
}

json_path = Path(OUTPUT_DIR) / f"{RUN_NAME}_metrics.json"
json_path.write_text(json.dumps(metrics_payload, indent=2, ensure_ascii=False))
print("JSON 저장:", json_path)

csv_rows = []
for scope, data in (("val", per_class), ("test", test_per_class)):
    for row in data:
        csv_rows.append({"scope": scope, **row})
for scope, data in (("val", overall), ("test", test_overall)):
    if data:
        csv_rows.append({"scope": scope, "class": "ALL", "class_ko": "전체", **data})
csv_path = Path(OUTPUT_DIR) / f"{RUN_NAME}_metrics.csv"
pd.DataFrame(csv_rows).to_csv(csv_path, index=False, encoding="utf-8-sig")
print("CSV 저장:", csv_path)
print()
print(json.dumps({"val_overall": overall, "test_overall": test_overall}, indent=2))

## 16. ONNX 변환 및 내려받기

변환된 `best.onnx` 를 로컬 저장소의 `models/best.onnx` 로 옮긴다.

In [ ]:
onnx_path = best_model.export(format="onnx", imgsz=CFG["imgsz"], opset=12, simplify=False)
onnx_path = Path(onnx_path)
print("ONNX 생성:", onnx_path, f"({onnx_path.stat().st_size/1e6:.1f} MB)")

# 확인: onnxruntime 으로 로드되는지 검증
import onnxruntime as ort
sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
print("입력:", [(i.name, i.shape) for i in sess.get_inputs()])
print("출력:", [(o.name, o.shape) for o in sess.get_outputs()])
print("메타데이터 names:", sess.get_modelmeta().custom_metadata_map.get("names"))

final_onnx = Path(OUTPUT_DIR) / "best.onnx"
shutil.copy2(onnx_path, final_onnx)
if DRIVE_MOUNTED:
    drive_out = Path("/content/drive/MyDrive/roadlens_anyang")
    drive_out.mkdir(parents=True, exist_ok=True)
    shutil.copy2(onnx_path, drive_out / "best.onnx")
    shutil.copy2(best_pt, drive_out / "best.pt")
    shutil.copy2(json_path, drive_out / json_path.name)
    shutil.copy2(csv_path, drive_out / csv_path.name)
    print("Drive 에 복사:", drive_out)

print("\n다음 단계: 이 파일을 로컬 저장소의 models/best.onnx 로 복사하세요.")

In [ ]:
# 브라우저로 직접 내려받기 (Colab 전용)
try:
    from google.colab import files
    files.download(str(final_onnx))
    files.download(str(json_path))
    files.download(str(csv_path))
except Exception as e:
    print("자동 다운로드를 사용할 수 없습니다:", e)
    print("Drive 또는 파일 탐색기에서 직접 내려받으세요:", final_onnx)

## 17. 마무리 점검

- [ ] `outputs/*_metrics.json` 의 mAP·Precision·Recall 을 README 의 시험 결과표에 **그대로** 옮겼는가
- [ ] `best.onnx` 를 로컬 `roadlens-anyang/models/best.onnx` 에 배치했는가
- [ ] `RUN_MODE` 가 무엇이었는지 기록했는가 (smoke_test 결과는 성능 근거로 쓸 수 없다)
- [ ] 분할 방식(`split_strategy`)이 `random_image` 라면 그 한계를 README 에 적었는가
- [ ] 사용한 국가 목록과 제외한 국가를 README 에 적었는가

### 로컬 앱에서 사용하기
```bash
cp ~/Downloads/best.onnx roadlens-anyang/models/best.onnx
cd roadlens-anyang
streamlit run app.py
```